In [ ]:
# Import necessary packages (might need to download some of these in the terminal)
import re
import pandas as pd
import nltk
import pdfplumber

In [ ]:
# Load the two datasets (change the directories)
df_minutes = pd.read_csv('/Users/emilymoore/Downloads/DS 4002 Project 1/meeting_mins/housing_uncertainty_results_with_score.csv')
df_caar = pd.read_csv('/Users/emilymoore/Downloads/DS 4002 Project 1/caar_market_reports/caar_2025_dashboard_metrics.csv')

# Standardize the date keys (YYYY-MM)
# The two datasets use different file naming conventions,
# so we extract and standardize a common monthly key for merging.
def get_minutes_key(name):
    match = re.search(r'(\d{2})-(\d{4})', str(name))
    return f"{match.group(2)}-{match.group(1)}" if match else None

def get_caar_key(name):
    # Extracts YY-MM from the start of the file name
    match = re.search(r'^(\d{2})-(\d{2})', str(name))
    return f"20{match.group(1)}-{match.group(2)}" if match else None

# Apply the key-extraction functions to create a standardized month identifier
df_minutes['month_key'] = df_minutes['file_name'].apply(get_minutes_key)
df_caar['month_key'] = df_caar['file_name'].apply(get_caar_key)

# Calculate Uncertainty Score (if not already in the file)
# Define uncertainty as the proportion of housing-related sentences
# that contain uncertainty-related language
# This normalizes uncertainty by overall housing discussion intensity
df_minutes['uncertainty_score'] = (
    df_minutes['housing_uncertainty_sentences'] / df_minutes['housing_mentions']
).fillna(0) # Replace NaN values (e.g., division by zero) with 0

# Perform the Join
# We use an 'outer' join to keep all data, even if a month is missing from one file
df_combined = pd.merge(
    df_minutes, 
    df_caar, 
    on='month_key', # Join on standardized month identifier
    how='outer', # Keep all observations from both datasets
    suffixes=('_minutes', '_caar') # Distinguish overlapping column names
)

# Clean up and Sort
df_combined = df_combined.sort_values('month_key')

# Save the combined dataset for regression and visualization analysis
df_combined.to_csv('/Users/emilymoore/Downloads/DS 4002 Project 1/combined_housing_market_data.csv', index=False)

print(f"Joined successfully! Created {df_combined}")